# 📊 Contexto

Você foi contratado como Cientista de Dados para a rede de Farmácias  
**"Saúde é o que interessa"**. O produto mais crítico da loja possui uma demanda média diária estável, mas o estoque tem chegado a zero com frequência, causando perda de vendas e insatisfação dos clientes.

---

# 📦 Dados Iniciais do Problema

- **Estoque Inicial:** 500 unidades  
- **Demanda Diária:** Segue uma distribuição Normal (Gaussiana) com Média (μμ) = 30 unidades  
- **Lead Time (Tempo de Entrega):** O fornecedor leva exatamente 5 dias para entregar o produto após o pedido ser feito  
- **Lote de Compra:** Toda vez que um pedido é feito, compramos 400 unidades  
- **Regra de Negócio:** A decisão de compra deve considerar a **Posição de Estoque**  
  *(Estoque Físico + Pedidos que já foram feitos e ainda não chegaram)*

---

# 🧮 Parte 1 – Nível de Serviço

Utilizando uma **Variância (σσ²) de 2500**  
*(Desvio Padrão = 50 — `scale=50` em `random.normal` do numpy)*:

## 🔹 Tarefas

1. Simulem **1.000 cenários de 365 dias** cada  
2. Faça um **plot do estoque ao longo dos 365 dias**  
3. Encontrem qual o valor de **Ponto de Pedido (ROP)** necessário para garantir um  
   **Nível de Serviço entre 95% e 98%**  
   - Ou seja, o estoque físico só chega a zero em, no máximo, 5% das lojas/simulações  
   - O nível de serviço considera a proporção de cenários no qual o estoque chegou a zero  
4. Apresentem o **histograma do "Estoque Mínimo"** atingido nas simulações  

---

## ⚠️ Observação Importante

Um nível de serviço de **100%** indica que em nenhuma situação ocorreu a falta de estoque, porém é possível que você esteja mantendo um estoque muito alto.

- Exemplo:
  - Se toda vez que seu estoque chega a **450** você fizer um novo pedido, certamente nunca chegará ao zero, porém pode estar mantendo um estoque desnecessariamente alto.
  - Mas se você só fizer um pedido quando seu estoque chegar a **100**, certamente seu estoque esgotará antes do novo pedido chegar, e seu nível de serviço será de **0%**.

---

# 📉 Parte 2 – Variância Mais Alta

O cenário mudou. Devido a uma instabilidade no mercado:

- A média da demanda continua sendo **30**
- O desvio padrão aumentou para **150**

## 🔹 Tarefas

1. Mantendo o **ROP encontrado na Parte 1**, qual é o novo **Nível de Serviço**?  
2. Quanto vocês precisam **aumentar o ROP** para voltar à faixa de **95% a 98% de serviço**?

In [2]:
import numpy as np
import pandas as pd
import plotly.express as px

In [ ]:
numero_dias = 365
demanda_media = 30
sigma_media = 5 # Desvio padrão
numero_simulacoes = 10

lote = 400
estoque_inicial = 500
lead_time = 5

resultados_finais = []

In [ ]:

for s in range(numero_simulacoes):
    """
    loc: é a média da distribuição
    scale: é o desvio padrão dessa média
    size: é a quantidade de valores que você quer gerar
    """
    demanda = np.random.normal(loc=demanda_media, scale=sigma_media, size=numero_dias) 

demanda

array([37.31318844, 36.4163821 , 31.204273  , 37.84178314, 31.24255326,
       28.17103119, 38.77153177, 27.35474712, 32.53970296, 23.48275545,
       38.37796648, 32.06101407, 33.60883287, 30.69851874, 29.05686421,
       37.99783114, 31.2863657 , 30.20188924, 35.44033303, 26.44759533,
       33.97523486, 33.93823081, 26.68684641, 33.04528847, 36.00777209,
       33.52682763, 26.89670069, 26.1317824 , 19.51685423, 28.59709984,
       28.92333493, 29.69508221, 27.48219532, 27.01307789, 28.85044895,
       37.14537868, 22.88153977, 27.10901824, 17.07547312, 29.7077177 ,
       36.2223405 , 23.29347593, 34.36841776, 30.97073315, 33.26538905,
       34.29853782, 29.97016847, 22.38239292, 33.45830002, 29.71946481,
       34.2319758 , 33.11281568, 24.55779151, 24.09484807, 31.13386183,
       36.80382989, 33.82180192, 27.44759033, 21.22980951, 30.60899283,
       29.18101911, 31.6024929 , 25.63035747, 26.67344868, 43.01088514,
       34.41552019, 32.16918424, 25.06913971, 24.24725433, 29.65

In [ ]:
demanda = np.random.normal(loc=demanda_media, scale=sigma_media, size=numero_dias).astype(int)
demanda

array([32, 28, 21, 34, 29, 29, 32, 30, 28, 26, 29, 33, 31, 33, 31, 33, 34,
       35, 32, 30, 27, 29, 32, 24, 27, 26, 28, 29, 20, 26, 28, 34, 32, 20,
       34, 26, 22, 21, 37, 30, 26, 32, 30, 34, 28, 35, 30, 36, 25, 21, 21,
       33, 28, 18, 24, 26, 25, 19, 33, 40, 32, 28, 35, 32, 27, 34, 25, 33,
       25, 32, 35, 31, 22, 24, 23, 30, 34, 33, 26, 27, 27, 36, 30, 31, 26,
       35, 31, 26, 31, 37, 38, 26, 32, 28, 32, 31, 30, 38, 34, 36, 30, 23,
       26, 31, 30, 31, 26, 31, 38, 31, 25, 29, 28, 32, 26, 25, 29, 35, 28,
       31, 40, 25, 37, 28, 32, 27, 35, 38, 28, 32, 26, 25, 29, 31, 28, 33,
       33, 23, 32, 33, 28, 24, 36, 36, 31, 24, 32, 31, 26, 26, 35, 32, 35,
       22, 25, 39, 31, 38, 38, 26, 33, 29, 25, 26, 31, 37, 38, 32, 30, 28,
       20, 29, 34, 35, 30, 31, 31, 26, 33, 26, 39, 33, 38, 21, 33, 24, 21,
       33, 22, 39, 38, 29, 27, 41, 25, 24, 24, 32, 32, 35, 26, 34, 25, 30,
       28, 17, 29, 35, 32, 22, 25, 32, 27, 26, 41, 28, 19, 34, 28, 25, 35,
       33, 29, 24, 26, 32

esse distribuição não é o ideal para esse tipo de situação, mas por quê?

In [ ]:
for s in range(numero_simulacoes):
    demanda = np.random.normal(loc=demanda_media, scale=sigma_media, size=numero_dias)
    estoque_fisico = np.zeros(numero_dias)
    estoque_fisico[0] = estoque_inicial
demanda

array([33.60884402, 32.97284534, 27.52182255, 25.28864514, 36.68934736,
       34.85516087, 39.03539686, 32.81090436, 27.98419088, 30.89466752,
       23.51998103, 35.48161324, 21.81816196, 35.74037317, 37.11418896,
       24.36420889, 28.17531991, 31.57984607, 38.16665189, 28.59630405,
       23.33913112, 33.98906212, 17.82133111, 23.05435259, 21.92209435,
       26.03150921, 23.26790729, 33.0757192 , 27.42307104, 26.19746853,
       29.97939655, 31.41553644, 30.2875857 , 26.59815733, 25.91766285,
       46.89542864, 27.53116299, 22.87909857, 25.78088993, 30.37978699,
       33.36480532, 26.37964685, 35.13553171, 30.98118603, 34.73470356,
       33.18629054, 33.54629743, 28.53675049, 37.21677598, 30.89173129,
       35.66260489, 27.7028795 , 33.00520785, 32.91556   , 33.35673801,
       35.11375233, 25.98672533, 24.94729434, 29.98977826, 28.37781141,
       32.98669455, 32.31504392, 32.18229466, 21.72069803, 30.25419143,
       36.2617634 , 25.00129015, 27.67964494, 27.59477058, 25.53

In [36]:
rop = 150

In [26]:
for s in range(numero_simulacoes):
    demanda = np.random.normal(loc=demanda_media, scale=sigma_media, size=numero_dias)
    estoque_fisico = np.zeros(numero_dias)
    estoque_fisico[0] = estoque_inicial
    pedidos_futuros = [] # (d, q)

    for t in range(1, numero_dias):
        chegadas_hoje = sum(q for d, q in pedidos_futuros if d==t)
    
demanda

array([35.84448342, 32.49898053, 22.26490937, 26.72570489, 27.29757928,
       33.21784519, 25.88732586, 30.8504394 , 32.38259766, 26.9099237 ,
       33.21918005, 27.58658351, 32.52783888, 29.736474  , 40.08891332,
       32.03890563, 23.90668009, 27.32001887, 30.02755704, 29.72336622,
       24.37559316, 27.9495551 , 27.08704725, 28.84465988, 32.17392084,
       35.74791061, 31.22605425, 27.38961912, 36.06000853, 30.83646318,
       33.0492058 , 32.57955078, 26.85594084, 32.4204161 , 27.91792856,
       27.86215184, 36.81453603, 32.80504671, 28.30515686, 33.43980008,
       27.18286898, 39.25234584, 37.45249543, 30.13886843, 25.6092754 ,
       30.94809357, 28.7718133 , 27.97370737, 28.7528533 , 25.98725487,
       26.40243226, 33.58616722, 33.88383279, 32.30916739, 36.59291236,
       21.66844561, 27.47478699, 33.0533262 , 42.18640384, 31.0322931 ,
       29.53496577, 36.17711564, 32.03531868, 32.63442746, 24.18081768,
       25.85192913, 29.0604872 , 22.54949298, 28.00275439, 24.64

In [27]:
h = [(2, 30), (3, 40)]
h

[(2, 30), (3, 40)]

In [28]:
ch = sum(q for d, q in h if d==2)
ch

30

In [29]:
for s in range(numero_simulacoes):
    demanda = np.random.normal(loc=demanda_media, scale=sigma_media, size=numero_dias)
    estoque_fisico = np.zeros(numero_dias)
    estoque_fisico[0] = estoque_inicial
    pedidos_futuros = [] # (d, q)
    niveis_na_entrega = []

    for t in range(1, numero_dias):
        chegadas_hoje = sum(q for d, q in pedidos_futuros if d==t)

        if chegadas_hoje > 0:
            nivel_antes = estoque_fisico[t-1] - demanda[t]
            niveis_na_entrega.append(nivel_antes)

        estoque_fisico[t] = estoque_fisico[t-1] - demanda[t] + chegadas_hoje

        em_transito = sum(q for d, q in pedidos_futuros)
    
demanda

array([30.0726222 , 30.25682363, 34.62368008, 31.2274246 , 24.81177318,
       29.00483523, 30.14363445, 34.2238699 , 30.0446174 , 33.05114234,
       26.2640064 , 19.82011664, 30.00734545, 35.88926993, 39.61958117,
       32.25089868, 31.92295702, 33.8039088 , 30.90767927, 25.41460441,
       31.00159797, 29.54987374, 34.9714328 , 28.2617717 , 28.33600038,
       21.81271425, 25.5294    , 32.47804238, 34.04887323, 30.53355428,
       36.09190341, 27.05486588, 33.85885802, 25.14203313, 30.19000235,
       26.08992352, 30.30181493, 32.57984122, 40.05766167, 31.92496252,
       35.80259255, 27.33103064, 24.902158  , 26.56485362, 33.21999312,
       24.48922183, 30.09035259, 30.59016364, 34.11699514, 33.47098344,
       31.5293475 , 25.66544462, 30.93539681, 29.73318167, 25.95167577,
       24.54941339, 31.20561207, 32.80699263, 21.94045029, 33.42275998,
       35.27963812, 35.8024024 , 23.71364911, 38.03807773, 37.57025524,
       40.04393103, 30.730604  , 20.1812622 , 32.05339472, 30.50

In [ ]:
for s in range(numero_simulacoes):
    demanda = np.random.normal(loc=demanda_media, scale=sigma_media, size=numero_dias)
    estoque_fisico = np.zeros(numero_dias)
    estoque_fisico[0] = estoque_inicial
    pedidos_futuros = [] # (d, q)
    niveis_na_entrega = []

    for t in range(1, numero_dias):
        chegadas_hoje = sum(q for d, q in pedidos_futuros if d==t)

        if chegadas_hoje > 0:
            nivel_antes = estoque_fisico[t-1] - demanda[t]
            niveis_na_entrega.append(nivel_antes)

        estoque_fisico[t] = estoque_fisico[t-1] - demanda[t] + chegadas_hoje

        em_transito = sum(q for d, q in pedidos_futuros if d > t)
        posicao = estoque_fisico[t] + em_transito

        if posicao <= rop:
            pedidos_futuros.append((t + lead_time, lote))

    media_na_entrega = np.mean(niveis_na_entrega)
    media_na_entrega

In [ ]:
for s in range(numero_simulacoes):
    demanda = np.random.normal(loc=demanda_media, scale=sigma_media, size=numero_dias)
    estoque_fisico = np.zeros(numero_dias)
    estoque_fisico[0] = estoque_inicial
    pedidos_futuros = [] # (d, q)
    niveis_na_entrega = []

    for t in range(1, numero_dias):
        chegadas_hoje = sum(q for d, q in pedidos_futuros if d==t)

        if chegadas_hoje > 0:
            nivel_antes = max(estoque_fisico[t-1] - demanda[t] + chegadas_hoje, 0) # agora é o máximo
            niveis_na_entrega.append(nivel_antes)

        estoque_fisico[t] = max(estoque_fisico[t-1] - demanda[t] + chegadas_hoje, 0) # aqui também é máximo

        em_transito = sum(q for d, q in pedidos_futuros if d > t)
        posicao = estoque_fisico[t] + em_transito

        if posicao <= rop:
            pedidos_futuros.append((t + lead_time, lote))

    media_na_entrega = np.mean(niveis_na_entrega)
    resultados_finais.append({'estoque_minimo': np.min(estoque_fisico),
                              'estoque_media_na_entrega': media_na_entrega})
    
df = pd.DataFrame(resultados_finais)
df

,estoque_minimo,estoque_media_na_entrega
0,0.0,392.321084
1,0.0,388.147945
2,0.0,382.078449
3,0.0,384.866212
4,0.0,384.443169
5,0.0,387.196439
6,0.0,386.537339
7,0.0,383.337346
8,0.0,383.837585
9,0.0,382.310018


In [44]:
nivel_de_servico = (df['estoque_minimo'] == 0).mean()*100
nivel_de_servico

np.float64(100.0)

In [45]:
nivel_de_servico = (df['estoque_minimo'] != 0).mean()*100
nivel_de_servico

np.float64(0.0)

In [48]:
fig = px.histogram(df, x='estoque_minimo', nbins=30, template='plotly_white')
fig.show()

In [51]:
import plotly.graph_objects as go

fig_go = go.Figure(data=[go.Histogram(x=df['estoque_minimo', nbins=dict(start=0, end=50, size=1)])])
fig_go.show()

SyntaxError: invalid syntax. Maybe you meant '==' or ':=' instead of '='? (2388926920.py, line 3)